In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
# add the project root to the path so the src package can be imported
sys.path.append(os.path.abspath('..'))

import yaml
from ultralytics import YOLO
from src.utils import apply_smart_aug
from src.model_utils import smart_predict,export_enum_by_quad_using_model,analyze_quadrant_predictions,compare_best_vs_last
from src.model_callbacks import on_fit_epoch_end # It will ask you now immediatly (What Stage you are in right now?) Just Type your currently stage number
from src.generate_report import generate_report
import pandas as pd

%matplotlib inline

Patience Limit Is 20 Loaded Successfuly from the config file!


In [ ]:
with open('../configs/stage1.yaml','r') as f:

    stage1_config = yaml.safe_load(f)


with open('../configs/stage2.yaml','r') as f:

    stage2_config = yaml.safe_load(f)


with open('../configs/stage3.yaml','r') as f:

    stage3_config = yaml.safe_load(f)


with open(stage1_config['model_args']['data'],'r') as f:

    quad_data_yaml = yaml.safe_load(f)


with open(stage2_config['model_args']['data'],'r') as f:

    enum_data_yaml = yaml.safe_load(f)


with open(stage3_config['model_args']['data'],'r') as f:

    dis_data_yaml = yaml.safe_load(f)

In [3]:
stage1_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant/train_quadrant.json',
  's1_main_path': '..\\Data\\Processed\\Stage 1 (Quadrant Detection)',
  'runs_s1_output': '..\\Runs\\Stage 1'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 1 (Quadrant Detection)\\data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 50,
  'save': True,
  'imgsz': 1024,
  'batch': 8,
  'patience': 10,
  'optimizer': 'AdamW',
  'lr0': 0.001,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 1',
  'save_dir': '..\\Runs\\Stage 1',
  'auto_augment': 'None',
  'augment': True,
  'mosaic': 0.0,
  'mixup': 0.0,
  'copy_paste': 0.0,
  'cutmix': 0.0,
  'hsv_h': 0.015,
  'hsv_s': 0.4,
  'hsv_v': 0.4,
  'degrees': 5.0,
  'translate': 0.05,
  'scale': 0.1,
  'shear': 0.0,
  'perspective': 0.0,
  'flipud': 0.0

In [4]:
stage2_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration/train_quadrant_enumeration.json',
  's2_main_path': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)',
  'runs_s2_output': '..\\Runs\\Stage 2',
  'diagnosis_quadrants_train_path': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)\\Diagnosis Quadrants Train',
  'runs_s2_continued_output': '..\\Runs\\Stage 2 Continued'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 2 (Enumeration Detection)\\data.yaml',
  'model': '../Models/yolo26s.pt',
  'epochs': 100,
  'save': True,
  'imgsz': 1024,
  'batch': 8,
  'patience': 20,
  'optimizer': 'AdamW',
  'lr0': 0.005,
  'lrf': 0.01,
  'plots': True,
  'verbose': True,
  'device': 'cuda',
  'workers': 4,
  'project': 'Runs',
  'name': 'Stage 2',
  'save_dir': '..\\Runs\\Stage 2',
  'auto_augment': 'None',
  'augment': False,
  'mo

In [5]:
stage3_config

{'paths': {'original_images_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/xrays',
  'original_json_path': '../Data/Raw/DENTEX CHALLENGE 2023/Training_data/quadrant-enumeration-disease/train_quadrant_enumeration_disease.json',
  's3_main_path': '..\\Data\\Processed\\Stage 3 (Disease Classifier)',
  'runs_s3_output': '..\\Runs\\Stage 3'},
 'model_args': {'data': '..\\Data\\Processed\\Stage 3 (Disease Classifier)\\data.yaml'}}

In [6]:
quad_data_yaml

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\train\\images',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\valid\\images',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 1 (Quadrant Detection)\\test\\images',
 'nc': 4,
 'names': ['Upper Right', 'Upper Left', 'Lower Left', 'Lower Right']}

In [7]:
enum_data_yaml

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\train\\images',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\valid\\images',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 2 (Enumeration Detection)\\test\\images',
 'nc': 8,
 'names': [0, 1, 2, 3, 4, 5, 6, 7]}

In [8]:
dis_data_yaml 

{'train': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\train\\',
 'val': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\valid\\',
 'test': 'C:\\Users\\ibrah.HIMA\\OneDrive\\Desktop\\Full AI\\Projects\\GitHub - Kaggle Projects\\dental-xray-ai\\Data\\Processed\\Stage 3 (Disease Classifier)\\test\\',
 'nc': 5,
 'names': ['Impacted', 'Caries', 'Periapical', 'Deep Caries', 'No Disease']}

In [9]:
# load the base checkpoint specified in the config, this is the starting point before training
yolo_model = YOLO(stage2_config['model_args']['model'])

In [ ]:
yolo_model.add_callback('on_fit_epoch_end',on_fit_epoch_end)

# training already ran once; left here commented out for reference
yolo_model.train(**stage2_config['model_args'])

New https://pypi.org/project/ultralytics/8.4.105 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.83  Python-3.13.12 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=None, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\Data\Processed\Stage 2 (Enumeration Detection)\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=768, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=../Models/yolo26s.pt

In [ ]:
compare_best_vs_last(os.path.join(stage2_config['paths']['runs_s2_output'], 'results.csv'))

In [ ]:
# load the checkpoint saved at the end of quadrant training, this is the model we'll actually use for inference
quadrant_model_detection = YOLO(os.path.join(stage1_config['paths']['runs_s1_output'], 'weights', 'last.pt'))

# load the checkpoint saved at the best of teeth training, this is the model we'll actually use for inference
teeth_model_detection = YOLO(os.path.join(stage2_config['paths']['runs_s2_output'], 'weights', 'best.pt'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'])

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],save_output=True,save_dir=os.path.join(stage2_config['paths']['runs_s2_output'],'Test Outputs Predictions'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],save_crop_output_image=True,save_dir=os.path.join(stage2_config['paths']['runs_s2_output'],'Test Outputs Predictions'))

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True)

In [ ]:
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True,apply_custom_draw_box=True)

In [ ]:
# overlay the ground-truth boxes next to the predictions for a visual comparison
smart_predict(teeth_model_detection,enum_data_yaml['test'],show_true_boxes=True)

In [11]:
# these pickles already have the train/valid/test split baked in
train_dis_df = pd.read_pickle(os.path.join(stage3_config['paths']['s3_main_path'], 'train_df (splitted).pkl'))
valid_dis_df = pd.read_pickle(os.path.join(stage3_config['paths']['s3_main_path'], 'valid_df (splitted).pkl'))
test_dis_df = pd.read_pickle(os.path.join(stage3_config['paths']['s3_main_path'], 'test_df (splitted).pkl'))

print('Train Rows:',train_dis_df.shape[0])
print('Valid Rows:',valid_dis_df.shape[0])
print('Test Rows:',test_dis_df.shape[0])

Train Rows: 2835
Valid Rows: 346
Test Rows: 324


In [12]:
train_dis_df.head(3)

,File_Name,Bbox,Height,Width,Disease_Name,Quad,Enumeration
0,train_673.png,"[542.0, 698.0, 220.0, 271.0]",1316,2744,impacted,Lower Right,7
1,train_673.png,"[1952.0, 693.0, 177.0, 270.0]",1316,2744,impacted,Lower Left,7
2,train_673.png,"[675.0, 708.0, 243.0, 300.0]",1316,2744,caries,Lower Right,6
